In [1]:


import pandas as pd
import numpy as np
from datetime import timedelta

# ============================================================
# 参数设置
# ============================================================

# 特征时滞（分钟）
FEATURE_LAG = 5  # X使用t-10分钟及之前的数据

# LULD阈值
LULD_THRESHOLD = 0.10  # 10%

# Price-Volume异常阈值
PRICE_Z_THRESHOLD = 2.0
VOLUME_Z_THRESHOLD = 2.5

# 滚动窗口
ROLLING_WINDOW = 12  # 60分钟

# ============================================================
# Step 1: 检测所有episodes（时间t）
# ============================================================

def detect_all_episodes(df):
    """
    在时间t检测所有manipulation signals
    允许使用price/volume，因为这是Y label
    """
    
    print("\n" + "="*70)
    print("DETECTING ALL EPISODES AT TIME T")
    print("="*70)
    
    df = df.copy()
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # 计算returns和z-scores (at time t)
    df['returns'] = df['close'].pct_change()
    df['returns_mean'] = df['returns'].rolling(ROLLING_WINDOW, min_periods=1).mean()
    df['returns_std'] = df['returns'].rolling(ROLLING_WINDOW, min_periods=1).std()
    df['volume_mean'] = df['volume'].rolling(ROLLING_WINDOW, min_periods=1).mean()
    df['volume_std'] = df['volume'].rolling(ROLLING_WINDOW, min_periods=1).std()
    
    df['returns_z'] = (df['returns'] - df['returns_mean']) / (df['returns_std'] + 1e-8)
    df['volume_z'] = (df['volume'] - df['volume_mean']) / (df['volume_std'] + 1e-8)
    
    # LULD detection (at time t)
    df['is_luld'] = (df['returns'].abs() > LULD_THRESHOLD).astype(int)
    
    # Price-volume anomaly (at time t)
    df['is_price_spike'] = (df['returns_z'].abs() > PRICE_Z_THRESHOLD).astype(int)
    df['is_volume_surge'] = (df['volume_z'] > VOLUME_Z_THRESHOLD).astype(int)
    df['is_anomaly'] = ((df['is_price_spike'] == 1) & (df['is_volume_surge'] == 1)).astype(int)
    
    # 合并所有signals
    df['is_episode_t'] = ((df['is_luld'] == 1) | (df['is_anomaly'] == 1)).astype(int)
    
    episode_count = df['is_episode_t'].sum()
    
    print(f"\nDetected episodes at time t:")
    print(f"  LULD events: {df['is_luld'].sum():,}")
    print(f"  Price-Volume anomalies: {df['is_anomaly'].sum():,}")
    print(f"  Total unique episodes: {episode_count:,}")
    
    return df


# ============================================================
# Step 2: 创建Time-Lagged Features
# ============================================================

def create_lagged_features(df, lag_minutes=10):
    """
    创建滞后features
    
    时间t的X features = 时间(t-lag_minutes)的market data
    时间t的Y label = 时间t的episode detection
    
    这样X和Y之间有lag_minutes的时间gap，没有overlap
    """
    
    print("\n" + "="*70)
    print(f"CREATING TIME-LAGGED FEATURES (LAG={lag_minutes} minutes)")
    print("="*70)
    
    # 计算lag的bar数量（5分钟1个bar）
    lag_bars = lag_minutes // 5
    
    print(f"\nLag configuration:")
    print(f"  Lag time: {lag_minutes} minutes")
    print(f"  Lag bars: {lag_bars}")
    
    # Market features在t时刻使用的是(t-lag)的值
    lagged_features = ['close', 'volume', 'returns', 'high', 'low']
    
    for feat in lagged_features:
        if feat in df.columns:
            df[f'{feat}_lag{lag_minutes}min'] = df[feat].shift(lag_bars)
    
    # 滞后的z-scores
    df[f'returns_z_lag{lag_minutes}min'] = df['returns_z'].shift(lag_bars)
    df[f'volume_z_lag{lag_minutes}min'] = df['volume_z'].shift(lag_bars)
    
    print(f"\n✓ Created {len(lagged_features)+2} lagged features")
    
    return df


# ============================================================
# Step 3: 创建EWS Labels
# ============================================================

def create_ews_labels_lagged(df, windows=[5, 15, 30], lag_minutes=10):
    """
    创建EWS labels with time lag
    
    关键:
    - EWS@T label在时间t
    - 但X features使用(t-lag)的数据
    - 所以预测窗口实际是[t-T-lag, t-lag]
    """
    
    print("\n" + "="*70)
    print("CREATING EWS LABELS WITH TIME LAG")
    print("="*70)
    
    for T in windows:
        df[f'EWS_{T}min'] = 0
    
    episode_indices = df[df['is_episode_t'] == 1].index
    
    if len(episode_indices) == 0:
        print("  WARNING: No episodes detected!")
        return df
    
    for ep_idx in episode_indices:
        ep_time = df.loc[ep_idx, 'timestamp']
        
        for T in windows:
            # 预警窗口: [ep_time - T, ep_time]
            # 但features用的是lagged data
            start_time = ep_time - timedelta(minutes=T)
            
            mask = (df['timestamp'] >= start_time) & (df['timestamp'] <= ep_time)
            df.loc[mask, f'EWS_{T}min'] = 1
    
    # 统计
    print(f"\nLabel distribution:")
    for T in windows:
        col = f'EWS_{T}min'
        pos = df[col].sum()
        pct = pos / len(df) * 100 if len(df) > 0 else 0
        print(f"  {col}: {pos:>6,} positive ({pct:>5.2f}%)")
    
    print(f"\nTime lag verification:")
    print(f"  X features use data from: t-{lag_minutes} minutes")
    print(f"  Y labels defined at: t")
    print(f"  Temporal gap: {lag_minutes} minutes (no overlap!)")
    
    return df


# ============================================================
# Main
# ============================================================

def main():
    print("\n" + "="*70)
    print("TIME-LAGGED Y LABEL GENERATION")
    print("="*70)
    
    print("\nApproach:")
    print("  ✓ Detect episodes at time t (using price/volume)")
    print("  ✓ Features use data from (t-lag)")
    print("  ✓ Temporal separation prevents leakage")
    print("  ✓ More episodes than halts-only approach")
    
    # Load data
    print("\n" + "="*70)
    print("STEP 1: LOADING DATA")
    print("="*70)
    
    try:
        df = pd.read_csv('gme_5min.csv')
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        if df['timestamp'].dt.tz is not None:
            df['timestamp'] = df['timestamp'].dt.tz_localize(None)
        
        print(f"\n✓ Loaded {len(df):,} bars")
    except FileNotFoundError:
        print("\n✗ Error: gme_5min.csv not found!")
        return
    
    # Detect episodes
    print("\n" + "="*70)
    print("STEP 2: DETECTING EPISODES")
    print("="*70)
    
    df = detect_all_episodes(df)
    
    # Create lagged features
    print("\n" + "="*70)
    print("STEP 3: CREATING LAGGED FEATURES")
    print("="*70)
    
    df = create_lagged_features(df, lag_minutes=FEATURE_LAG)
    
    # Create EWS labels
    print("\n" + "="*70)
    print("STEP 4: CREATING EWS LABELS")
    print("="*70)
    
    df = create_ews_labels_lagged(df, windows=[5, 15, 30], lag_minutes=FEATURE_LAG)
    
    # 移除前面lag_bars行（没有完整lagged features）
    lag_bars = FEATURE_LAG // 5
    df = df.iloc[lag_bars:].reset_index(drop=True)
    
    # Save
    print("\n" + "="*70)
    print("STEP 5: SAVING")
    print("="*70)
    
    output_file = 'gme_final_with_EWS_labels_TIME_LAGGED.parquet'
    df.to_parquet(output_file, index=False)
    
    print(f"\n✓ Saved to: {output_file}")
    print(f"  Total bars: {len(df):,}")
    
    # Summary
    print("\n" + "="*70)
    print("SUMMARY - TIME-LAGGED VERSION")
    print("="*70)
    
    print(f"\nConfiguration:")
    print(f"  Feature lag: {FEATURE_LAG} minutes")
    print(f"  Episode detection: LULD + Price-Volume anomaly")
    print(f"  Total episodes: {df['is_episode_t'].sum():,}")
    
    print(f"\nKey features (use these as X):")
    print(f"  - close_lag{FEATURE_LAG}min")
    print(f"  - volume_lag{FEATURE_LAG}min")
    print(f"  - returns_z_lag{FEATURE_LAG}min")
    print(f"  - etc.")
    
    print(f"\nY labels:")
    for T in [5, 15, 30]:
        col = f'EWS_{T}min'
        pos = df[col].sum()
        print(f"  {col}: {pos:,} positive")
    
    print("\n✓ NO DATA LEAKAGE: X uses (t-lag), Y uses t")


if __name__ == "__main__":
    main()


TIME-LAGGED Y LABEL GENERATION

Approach:
  ✓ Detect episodes at time t (using price/volume)
  ✓ Features use data from (t-lag)
  ✓ Temporal separation prevents leakage
  ✓ More episodes than halts-only approach

STEP 1: LOADING DATA

✓ Loaded 59,147 bars

STEP 2: DETECTING EPISODES

DETECTING ALL EPISODES AT TIME T

Detected episodes at time t:
  LULD events: 119
  Price-Volume anomalies: 592
  Total unique episodes: 693

STEP 3: CREATING LAGGED FEATURES

CREATING TIME-LAGGED FEATURES (LAG=5 minutes)

Lag configuration:
  Lag time: 5 minutes
  Lag bars: 1

✓ Created 7 lagged features

STEP 4: CREATING EWS LABELS

CREATING EWS LABELS WITH TIME LAG

Label distribution:
  EWS_5min:  1,281 positive ( 2.17%)
  EWS_15min:  2,391 positive ( 4.04%)
  EWS_30min:  3,989 positive ( 6.74%)

Time lag verification:
  X features use data from: t-5 minutes
  Y labels defined at: t
  Temporal gap: 5 minutes (no overlap!)

STEP 5: SAVING

✓ Saved to: gme_final_with_EWS_labels_TIME_LAGGED.parquet
  Tot

In [3]:
"""
将官方 Trade Halts 补充进 EWS@T 标签
======================================
逻辑:
  新 is_episode_t = 原 is_episode_t OR 官方halt期间

然后基于新的 is_episode_t 重新生成:
  EWS_5min, EWS_15min, EWS_30min

覆盖保存原文件，不改名。
"""

import pandas as pd
from datetime import timedelta

# ─── 1. 读取文件 ──────────────────────────────────────────────────────────────
df    = pd.read_parquet('gme_final_with_EWS_labels_TIME_LAGGED.parquet')
halts = pd.read_csv('Trade_halts_Historical.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'])
halts = halts[halts['Symbol'] == 'GME'].copy()

print(f"原始 is_episode_t=1: {int(df['is_episode_t'].sum())}")
print(f"官方 halt 记录数   : {len(halts)}")

# ─── 2. 将官方 halt 标记进 is_episode_t ─────────────────────────────────────
halts['halt_dt']   = pd.to_datetime(halts['Halt Date'].astype(str) + ' ' +
                                    halts['Halt Time'].astype(str))
halts['resume_dt'] = pd.to_datetime(halts['Resume Date'].astype(str) + ' ' +
                                    halts['NYSE Resume Time'].astype(str))

# 新增一列记录是否来自官方halt（方便后续区分）
df['is_official_halt'] = 0


for _, row in halts.iterrows():
    mask = (df['timestamp'] >= row['halt_dt']) & \
           (df['timestamp'] <= row['resume_dt'] + pd.Timedelta(minutes=5))
    df.loc[mask, 'is_official_halt'] = 1
    df['is_official_halt_lag1'] = df['is_official_halt'].shift(1)  # 上一个bar是否在halt
    df['is_official_halt_lag2'] = df['is_official_halt'].shift(2)  # 上两个bar

# 合并：原有 OR 官方halt
df['is_episode_t'] = ((df['is_episode_t'] == 1) |
                      (df['is_official_halt'] == 1)).astype(int)

print(f"合并后 is_episode_t=1: {int(df['is_episode_t'].sum())}  "
      f"(新增 {int(df['is_official_halt'].sum())} 个官方halt bars)")

# ─── 3. 重新生成 EWS labels ──────────────────────────────────────────────────
for T in [5, 15, 30]:
    df[f'EWS_{T}min'] = 0

episode_indices = df[df['is_episode_t'] == 1].index

for ep_idx in episode_indices:
    ep_time = df.loc[ep_idx, 'timestamp']
    for T in [5, 15, 30]:
        start_time = ep_time - timedelta(minutes=T)
        mask = (df['timestamp'] >= start_time) & (df['timestamp'] <= ep_time)
        df.loc[mask, f'EWS_{T}min'] = 1

# ─── 4. 统计对比 ─────────────────────────────────────────────────────────────
print(f"\n更新后 EWS 正样本数:")
for col in ['EWS_5min', 'EWS_15min', 'EWS_30min']:
    print(f"  {col}: {int(df[col].sum())}")

# ─── 5. 保存（覆盖原文件）───────────────────────────────────────────────────
output = 'gme_final_with_EWS_labels_TIME_LAGGED.parquet'
df.to_parquet(output, index=False)
print(f"\n✓ 已保存: {output}")

原始 is_episode_t=1: 805
官方 halt 记录数   : 65
合并后 is_episode_t=1: 805  (新增 120 个官方halt bars)

更新后 EWS 正样本数:
  EWS_5min: 1417
  EWS_15min: 2565
  EWS_30min: 4199

✓ 已保存: gme_final_with_EWS_labels_TIME_LAGGED.parquet
